# `awesomePMC` — Quickstart

This notebook walks through the three layers of the package:

1. **`pmcprg.copulas`** — bivariate copula families (PDF / CDF / sampling / fitting).
2. **`pmcprg.pmc`** — Pairwise Markov Chain models with copula-based transitions: simulation, MPM classification, and **ICE / SEM** unsupervised estimation.
3. **`pmcprg.diagnostics`** — model-agnostic goodness-of-fit tools (multivariate Kolmogorov–Smirnov test).

Run the cells top-to-bottom. Estimated runtime: < 1 minute end-to-end.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import pmcprg
print(f"awesomePMC v{pmcprg.__version__}")

---
## 1. Copula basics

Every copula in `pmcprg.copulas` is parameterised by Kendall's τ. Multi-parameter families (Student, BB1) accept their second parameter as a kwarg.

In [ ]:
from pmcprg.copulas import CopulaGaussian, CopulaClayton, CopulaGH, CopulaFrank

cop = CopulaGaussian(tau_k=0.6)
print("PDF (0.3, 0.7):", cop.pdf([0.3, 0.7]))
print("CDF (0.3, 0.7):", cop.cdf([0.3, 0.7]))
print("h(0.7 | 0.3) :", cop.conditional_cdf(0.7, 0.3))
print("Tail (λ_L, λ_U):", cop.tail_dependence())

### Vectorised PDF on an array of pseudo-observations

`pdf_array` is the hot path used by forward-backward and ICE. It's 50–100× faster than a Python loop on `pdf`.

In [ ]:
rng = np.random.default_rng(0)
uv  = np.column_stack([rng.uniform(0.05, 0.95, 1000),
                       rng.uniform(0.05, 0.95, 1000)])

%timeit -n 5 -r 3 [cop.pdf([float(u), float(v)]) for u, v in uv]
%timeit -n 5 -r 3 cop.pdf_array(uv)

### Sampling and fitting

Sampling uses the Rosenblatt inverse h-function (closed form for Gaussian / Clayton / Frank, Brent fallback elsewhere).

In [ ]:
samples = CopulaClayton(tau_k=0.5).sample(n=2000, seed=42)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(samples[:, 0], samples[:, 1], s=4, alpha=0.5)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
ax.set_title("Clayton τ=0.5 — n=2000 samples")
ax.set_xlabel("u"); ax.set_ylabel("v")
plt.show()

In [ ]:
# Fit a Clayton copula to the same samples
fit = CopulaClayton.fit(samples, method="mle")
print(fit)
print(f"AIC: {fit.aic:.2f}   BIC: {fit.bic:.2f}")

### Family selection on the same data

AIC ranks the families on a common log-likelihood scale; smaller is better. Clayton should win since the data was generated from Clayton.

In [ ]:
candidates = [CopulaGaussian, CopulaClayton, CopulaGH, CopulaFrank]

rows = []
for cls in candidates:
    r = cls.fit(samples, method="mle")
    rows.append((cls.__name__, r.tau_k, r.log_likelihood, r.aic))

print(f"{'Family':<18s} {'τ̂':>8s} {'loglik':>10s} {'AIC':>10s}")
for name, t, ll, aic in sorted(rows, key=lambda r: r[3]):
    print(f"{name:<18s} {t:>8.3f} {ll:>10.2f} {aic:>10.2f}")

---
## 2. Bivariate joint law (Sklar)

`BivariateLaw` couples a copula with two scipy.stats marginal distributions:
$$f(x, y) = f_1(x) \, f_2(y) \, c(F_1(x), F_2(y))$$

In [ ]:
from scipy.stats import norm, expon
from pmcprg.copulas import BivariateLaw

law = BivariateLaw(
    copula       = CopulaGaussian(tau_k=0.4),
    left_margin  = (norm,  0.0, 1.0),    # standard normal
    right_margin = (expon, 0.0, 2.0),    # exp(scale=2)
)
law.set_seed(42)
data = law.sample(2000)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(data[:, 0], data[:, 1], s=4, alpha=0.4)
ax.set_xlabel("X (Normal)"); ax.set_ylabel("Y (Exponential)")
ax.set_title("Sklar bivariate sample — Gaussian copula τ=0.4")
plt.show()

---
## 3. PMC models — simulation

Models are TOML files; the package ships five demo models in `pmcprg/pmc/models/`. Five variants are supported:

| Variant   | Prior | Margins      | Copula |
|-----------|-------|--------------|--------|
| HMC-IN    | A     | per class    | no     |
| HMC-IN2   | A     | pair (i,j)   | no     |
| HMC-DN    | A     | pair (i,j)   | yes    |
| PMC-IN    | p     | pair (i,j)   | no     |
| PMC       | p     | pair (i,j)   | yes    |

In [ ]:
from pathlib import Path
from pmcprg.pmc import PMCModel, simulate, classify, ice, error_rate

MODELS_DIR = Path(pmcprg.__file__).parent / "pmc" / "models"
for f in sorted(MODELS_DIR.glob("*.toml")):
    print(" ", f.name)

In [ ]:
mdl = PMCModel(MODELS_DIR / "pmc_gauss_k2.toml")
print(mdl)
print(f"  variant.uses_copula = {mdl.variant.uses_copula}")
print(f"  variant.has_markov_prior = {mdl.variant.has_markov_prior}")
print(f"  π = {mdl.stationary_pi.round(3)}")
print(f"  p =\n{mdl.prior_p.round(3)}")

In [ ]:
X_ref, Y = simulate(mdl, N=2000, seed=0)

fig, ax = plt.subplots(figsize=(10, 3))
for k in range(mdl.K):
    mask = X_ref == k
    ax.scatter(np.arange(len(Y))[mask], Y[mask], s=3, alpha=0.6, label=f"X={k}")
ax.set_xlabel("n"); ax.set_ylabel("Y")
ax.set_title(f"Simulated sequence — {mdl.name} (N={len(Y)})")
ax.legend(markerscale=3)
plt.show()

---
## 4. Supervised classification (MPM)

Forward-backward → posterior marginals γ_n(j) → MPM `argmax`.  
We pass the **same** model used for simulation, so error should be near the Bayes rate.

In [ ]:
X_hat, gamma, log_lik = classify(mdl, Y)
er = error_rate(X_ref, X_hat)
print(f"Log-likelihood : {log_lik:.2f}")
print(f"Error rate     : {er:.4f}  ({er*100:.1f} %)")

fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
axes[0].plot(gamma[:, 0], lw=0.5, label=r"$\gamma_n(X{=}0|Y)$")
axes[0].plot(gamma[:, 1], lw=0.5, label=r"$\gamma_n(X{=}1|Y)$")
axes[0].set_ylabel("Posterior"); axes[0].set_ylim(-0.05, 1.05); axes[0].legend()
axes[1].fill_between(np.arange(len(X_ref)), (X_ref != X_hat).astype(int),
                     alpha=0.4, color="red")
axes[1].set_xlabel("n"); axes[1].set_ylabel("errors")
plt.suptitle("MPM classification — posterior marginals (top) and errors (bottom)")
plt.tight_layout()
plt.show()

---
## 5. Unsupervised estimation (ICE)

Iterative Conditional Estimation: **without** access to `X_ref`, recover the model parameters from `Y` alone. We start from a deliberately perturbed version of the true model and watch ICE converge.

In [ ]:
# Perturbed initial model: shift τ on every copula by -0.2
raw = mdl.raw
for blk in raw.get("copulas", []):
    blk["tau"] = max(-0.9, blk["tau"] - 0.2)
init = PMCModel.from_dict(raw)

fitted, trace = ice(init, Y, ice_cfg={
    "max_iter":   15,
    "candidates": ["Gauss", "Clayton", "GH", "Frank"],
})

# `trace` exposes per-iteration diagnostics consumed by the GUI's View panel
# (log_liks, tau_history, family_history, p_history, margin_history,
#  multistart_runs).
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(trace.log_liks, "o-")
ax.set_xlabel("ICE iteration"); ax.set_ylabel("log-likelihood")
ax.set_title("ICE convergence"); ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Compare fitted parameters to the ground truth
print("True copulas:")
for i in range(mdl.K):
    for j in range(mdl.K):
        cop = mdl.copula(i, j)
        print(f"  ({i},{j}): {cop.copula_enum.value.SHORT_NAME:>8s}  τ={cop.params['tau_k']:+.3f}")

print("\nFitted copulas:")
for i in range(fitted.K):
    for j in range(fitted.K):
        cop = fitted.copula(i, j)
        print(f"  ({i},{j}): {cop.copula_enum.value.SHORT_NAME:>8s}  τ={cop.params['tau_k']:+.3f}")

X_hat_fit, _, _ = classify(fitted, Y)
print(f"\nError rate with fitted model: {error_rate(X_ref, X_hat_fit):.4f}")

---
## 5b. SEM — Stochastic EM (sister estimator)

SEM shares ICE's M-step but completes the latent states with a single
Forward-Filter Backward-Sample draw $\tilde X \sim P(X \mid Y)$ at each
iteration. Its log-likelihood **fluctuates** around a stationary regime rather
than converging monotonically — for inference, average the post-burn-in
estimates. We reuse the **same** perturbed `init` and observations `Y`.

In [ ]:
from pmcprg.pmc import sem

# Same perturbed init and same Y as ICE above — only the estimator changes.
fitted_sem, trace_sem = sem(init, Y, sem_cfg={
    "max_iter":   15,
    "sem_seed":   0,
    "candidates": ["Gauss", "Clayton", "GH", "Frank"],
})

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(trace.log_liks,     "o-", label="ICE (deterministic)")
ax.plot(trace_sem.log_liks, "s-", alpha=0.75, label="SEM (stochastic)")
ax.set_xlabel("iteration"); ax.set_ylabel("log-likelihood")
ax.set_title("ICE vs SEM convergence"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

X_hat_sem, _, _ = classify(fitted_sem, Y)
print(f"SEM error rate with fitted model: {error_rate(X_ref, X_hat_sem):.4f}")

---
## 6. K=3 example — the `hmc_in_gauss_k3` model

The package now ships a K=3 demo. Note how `error_rate` automatically resolves arbitrary label permutations via the Hungarian algorithm.

In [ ]:
mdl3 = PMCModel(MODELS_DIR / "hmc_in_gauss_k3.toml")
X3_ref, Y3 = simulate(mdl3, N=3000, seed=1)
X3_hat, gamma3, ll3 = classify(mdl3, Y3)
print(f"K        : {mdl3.K}")
print(f"Log-lik  : {ll3:.2f}")
print(f"Error    : {error_rate(X3_ref, X3_hat):.4f}")

fig, ax = plt.subplots(figsize=(10, 3))
for k in range(mdl3.K):
    ax.hist(Y3[X3_ref == k], bins=40, alpha=0.5, label=f"X={k}", density=True)
ax.set_xlabel("Y"); ax.set_ylabel("density"); ax.legend()
ax.set_title("K=3 marginal histograms")
plt.show()

---
## 7. Goodness-of-fit — multivariate KS test (`pmcprg.diagnostics`)

The third layer is model-agnostic: it takes raw NumPy arrays and returns a
typed result. Here the two-sample multivariate Kolmogorov–Smirnov test checks
that observations **simulated from the ICE-fitted model** are distributionally
consistent with the real data — a good fit should **not** reject $H_0$.

In [ ]:
from pmcprg.diagnostics import mks_2samp

# Simulate from the ICE-fitted K=2 model (`fitted`, §5) and compare with the
# observed Y. A good fit should NOT reject H0. mks is O(N²) — subsample to stay
# fast; a 1-D Y is auto-reshaped to (N, 1).
_, Y_sim = simulate(fitted, N=len(Y), seed=7)
rng_mks  = np.random.default_rng(0)
n_cmp    = 300
obs = Y[rng_mks.choice(len(Y),     n_cmp, replace=False)]
sim = Y_sim[rng_mks.choice(len(Y_sim), n_cmp, replace=False)]
res = mks_2samp(obs, sim, alpha=0.05)
print(f"MKS 2-sample  N={n_cmp}  statistic={res.statistic:.4f}  "
      f"crit={res.critical_value:.4f}  reject H0={res.reject}")

---
## What next?

* `pmcprg.copulas`: each family has a `plot_overview(plot_dir)` that produces a 2×2 panel (PDF, CDF, h-function, samples). See `pmcprg/copulas/<family>.py` for runnable demos.
* **Real-data example**: `examples/uci_har_smartphone.ipynb` runs ICE + SEM + the MKS test on 3-D smartphone-accelerometer data (with an offline synthetic fallback).
* CLI: after `pip install -e .`, the `pmc` console script exposes the same workflow:
  ```
  pmc simulate       --model model.toml --N 5000 --out sim.csv
  pmc classify       --model model.toml --data sim.csv --out cls.csv
  pmc estimate       --model init.toml  --data sim.csv --algorithm {ice,sem} --out fitted.toml
  pmc classify-image --model model.toml --image img.png --out seg.png
  pmc estimate-image --model init.toml  --image img.png --algorithm {ice,sem} --out fitted.toml
  pmc gui            [model.toml]
  ```
* GUI: `pip install 'awesomepmc[gui]'` then `pmc gui pmcprg/pmc/models/pmc_gauss_k2.toml`.
* Tests: `pytest -q` runs the full suite; add `-m "not slow"` to skip the slow notebook / Monte-Carlo tests.